# Experiment 2 : Awake / N-REM / REM Classification

## Reading CSV files

In [ ]:
import pandas as pd

training_mice  = pd.read_csv('./pw3_data/EEG_mouse_data_1.csv')
training_mice2 = pd.read_csv('./pw3_data/EEG_mouse_data_2.csv')

training_data = pd.concat([training_mice, training_mice2], ignore_index=True)


## Choosing features

In [ ]:
feature_list = ["state"] + [f"amplitude_around_{i}_Hertz" for i in range(1, 26)]

input_training_mice = training_data[feature_list]
print(input_training_mice.head())

## Normalize and encode data

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

scaler  = StandardScaler()
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

state = pd.DataFrame(input_training_mice['state'])
encoder.fit(state)

output_training_mice = encoder.transform(state)
input_training_mice  = input_training_mice.drop(columns=['state'])

for column in input_training_mice:
    column_data = input_training_mice[column].to_frame()
    scaler.fit(column_data)
    input_training_mice[column] = scaler.transform(column_data)

## Create model and fold

In [ ]:
import keras
from keras import layers
from sklearn.model_selection import KFold

keras.utils.set_random_seed(123)
kf = KFold(n_splits=3, shuffle=True)

def create_model():
    mlp = keras.Sequential([
        layers.Input(shape=(25,)),
        layers.Dense(8, activation="relu"),
        layers.Dense(3, activation="softmax"),
    ])
    mlp.compile(
        optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.99),
        loss="categorical_crossentropy",
    )
    return mlp

mlp = create_model()
mlp.summary()

## Training

In [ ]:
import numpy as np

history_list = []
trained_mlp  = []

for i, (train_index, test_index) in enumerate(kf.split(input_training_mice)):
    mlp = create_model()
    x_train, x_test = input_training_mice.iloc[train_index], input_training_mice.iloc[test_index]
    y_train, y_test = np.array(output_training_mice)[train_index], np.array(output_training_mice)[test_index]

    history = mlp.fit(
        x=x_train, y=y_train,
        validation_data=(x_test, y_test),
        epochs=50
    )

    history_list.append(history)
    trained_mlp.append(mlp)

## Plot training history

In [ ]:
import matplotlib.pyplot as pl
%matplotlib inline

train_losses = np.array([history.history['loss']     for history in history_list])
val_losses   = np.array([history.history['val_loss'] for history in history_list])

mean_train_loss = np.mean(train_losses, axis=0)
std_train_loss  = np.std(train_losses,  axis=0)
mean_val_loss   = np.mean(val_losses,   axis=0)
std_val_loss    = np.std(val_losses,    axis=0)

pl.plot(mean_train_loss, label='Training Loss (Mean)')
pl.fill_between(range(len(mean_train_loss)), mean_train_loss - std_train_loss, mean_train_loss + std_train_loss, alpha=0.3, label='Training Loss (Std)')

pl.plot(mean_val_loss, label='Validation Loss (Mean)')
pl.fill_between(range(len(mean_val_loss)), mean_val_loss - std_val_loss, mean_val_loss + std_val_loss, alpha=0.3, label='Validation Loss (Std)')

pl.xlabel('Epochs')
pl.ylabel('Loss')
pl.legend()
pl.show()

## Performance

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns

def plot_confusion_matrix(confusion_matrix, title):
    pl.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix.astype(int), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["rem", "n-rem", "awake"], yticklabels=["rem", "n-rem", "awake"])
    pl.title(title)
    pl.xlabel('Predicted')
    pl.ylabel('True')
    pl.show()

mean_confusion_matrix = np.zeros((3, 3))
f1_scores_micro = []

for i, (train_index, test_index) in enumerate(kf.split(input_training_mice)):
    predictions = trained_mlp[i].predict(input_training_mice.iloc[test_index])
    true_labels = np.array(output_training_mice)[test_index]

    max_predictions = [np.argmax(p) for p in predictions]
    max_true_labels = [np.argmax(t) for t in true_labels]

    cm = confusion_matrix(max_true_labels, max_predictions)
    mean_confusion_matrix += cm
    plot_confusion_matrix(cm, f'Confusion Matrix - Fold {i + 1}')

    f1_per_class = f1_score(max_true_labels, max_predictions, average=None)
    f1_micro     = f1_score(max_true_labels, max_predictions, average='micro')
    f1_scores_micro.append(f1_micro)

    print(f"F1 per class - Fold {i+1}: {f1_per_class}")
    print(f"F1 micro     - Fold {i+1}: {f1_micro}")

plot_confusion_matrix(mean_confusion_matrix, 'Global confusion matrix')

print(f"Mean F1 micro across all folds: {np.mean(f1_scores_micro)}")